In [1]:
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder

from geoai.utils_ds.DataFrameOps import DataFrameOperations
df_ops = DataFrameOperations()


# READ DF

In [2]:
df = pd.read_csv("csv_files/dataset.csv")
print(df.head(5))
print(df.shape)

     BLUE   GREEN        RED     NIR       SWIR Landcover
0  2132.0  2566.0  2386.0000  2836.0  2376.0000   builtup
1  2132.0  2566.0  2386.0000  2836.0  2376.0000   builtup
2  2153.0  2605.0  2486.6667  2800.0  2518.5000   builtup
3  2172.0  2609.0  2494.5000  2780.0  2555.0000   builtup
4  2039.0  2458.0  2409.0000  2746.5  2449.3333   builtup
(1915, 6)


# SPLIT DATA

Data splitting involves dividing the dataset into separate subsets to train, validate, and test the model.
- Training set: Used to fit the model. It learns from this data.
- Validation set: Used to tune the parameters of the model. It helps in selecting the best model and configuring it properly.
- Test set: Used to assess the performance of the final model. It provides an unbiased evaluation.

In [3]:
X_train, X_val, X_test, y_train, y_val, y_test= df_ops.split_data(df, "Landcover")
X_train.to_csv("csv_files/X_train.csv", index=False)
X_val.to_csv("csv_files/X_val.csv", index=False)
X_test.to_csv("csv_files/X_test.csv", index=False)
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(1053, 5)
(479, 5)
(383, 5)


# FEATURE ENGINEERING

Feature engineering is the process of creating or improving features. It is more of a dark art than a science. Features are often created based on common sense, domain knowledge, or prior experience. There are certain common techniques for feature creation; however, there is no guarantee that creating new features will improve results. 

For example, we can compute for NDVI, REI, and NDBI which can be derived from raw satellite data and serve as important features that can enhance the analysis and interpretation of remote sensing data.

$$ NDVI = \frac{NIR - RED}{NIR + RED} $$
$$ NDBI = \frac{SWIR - NIR}{SWIR + NIR} $$
$$ REI = \frac{NIR - BLUE}{NIR + BLUE * NIR} $$

In [4]:
# compute ndvi
X_train["NDVI"] = (X_train["NIR"] - X_train["RED"]) / (X_train["NIR"] + X_train["RED"])
#compute ndbi
X_train["NDBI"] = (X_train["SWIR"] - X_train["NIR"]) / (X_train["SWIR"] + X_train["NIR"])
#compute rei
X_train["REI"] = (X_train["NIR"] - X_train["BLUE"]) / (X_train["NIR"] + X_train["BLUE"] * X_train["NIR"])
X_train.fillna(0, inplace=True)
#export 
X_train.to_csv("csv_files/X_train_with_indices.csv", index=False)
X_train.head(1)

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI
1553,1093.2,1193.3334,1184.3334,1759.0,2088.3333,0.195243,0.0856,0.000346


# Polynomial transform

In [5]:
numerical_columns = X_train.select_dtypes(include=['float64']).columns.tolist()
print(numerical_columns)
X_train = df_ops.polynomial_transform(X_train, numerical_columns, 2)
print(X_train.columns)


['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR', 'NDVI', 'NDBI', 'REI']
Index(['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR', 'NDVI', 'NDBI', 'REI', 'BLUE^2',
       'BLUE GREEN', 'BLUE RED', 'BLUE NIR', 'BLUE SWIR', 'BLUE NDVI',
       'BLUE NDBI', 'BLUE REI', 'GREEN^2', 'GREEN RED', 'GREEN NIR',
       'GREEN SWIR', 'GREEN NDVI', 'GREEN NDBI', 'GREEN REI', 'RED^2',
       'RED NIR', 'RED SWIR', 'RED NDVI', 'RED NDBI', 'RED REI', 'NIR^2',
       'NIR SWIR', 'NIR NDVI', 'NIR NDBI', 'NIR REI', 'SWIR^2', 'SWIR NDVI',
       'SWIR NDBI', 'SWIR REI', 'NDVI^2', 'NDVI NDBI', 'NDVI REI', 'NDBI^2',
       'NDBI REI', 'REI^2'],
      dtype='object')


# Binning/Discretization

 Separate feature values into several bins.

In [6]:
# Create binary NDVI
ndvi_binary_edges = [-float("inf"), 0.5, float("inf")]
ndvi_binary_labels = ["non_veg", "veg"]
X_train = df_ops.binarize_or_discretize(
    X_train, "NDVI", "NDVI_bin", ndvi_binary_edges, ndvi_binary_labels
)
X_train.head(1)

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,BLUE GREEN,...,SWIR NDVI,SWIR NDBI,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2,NDVI_bin
1553,1093.2,1193.3334,1184.3334,1759.0,2088.3333,0.195243,0.0856,0.000346,1195086.24,1.304552e+06,...,407.733421,178.762182,0.722406,0.03812,0.016713,0.000068,0.007327,0.00003,1.196637e-07,non_veg


In [7]:
# Create categorical NDVI
ndvi_category_edges = [-float("inf"), 0.2, 0.5, float("inf")]
ndvi_category_labels = ["low_veg", "medium_veg", "high_veg"]
X_train = df_ops.binarize_or_discretize(
    X_train, "NDVI", "NDVI_dis", ndvi_category_edges, ndvi_category_labels
)
X_train.head(1)

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,BLUE GREEN,...,SWIR NDBI,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2,NDVI_bin,NDVI_dis
1553,1093.2,1193.3334,1184.3334,1759.0,2088.3333,0.195243,0.0856,0.000346,1195086.24,1.304552e+06,...,178.762182,0.722406,0.03812,0.016713,0.000068,0.007327,0.00003,1.196637e-07,non_veg,low_veg


# APPLY ONE HOT ENCODING

Process used to convert categorical data into a binary (0,1) vector representation where each category is represented by a unique vector in the space.

In [8]:
X_train = df_ops.make_one_hot_encode(X_train, "NDVI_bin", X_train)
X_train.head(1)

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,BLUE GREEN,...,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2,NDVI_bin,NDVI_dis,NDVI_bin_non_veg,NDVI_bin_veg
1553,1093.2,1193.3334,1184.3334,1759.0,2088.3333,0.195243,0.0856,0.000346,1195086.24,1.304552e+06,...,0.03812,0.016713,0.000068,0.007327,0.00003,1.196637e-07,non_veg,low_veg,1,0


# Scale numeric values

In [9]:
numerical_columns = X_train.select_dtypes(include=['float64']).columns.tolist()
scaler = MinMaxScaler()
X_train_scaled = X_train.copy()
X_train_scaled[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_train.to_csv("csv_files/X_train_fe.csv", index=False)

# CONVERT CLASS STRING TO INTEGERS

In [ ]:
y_train

1553       road
398     builtup
441     builtup
1341       road
70      builtup
         ...   
673     builtup
736     builtup
1490       road
1212       road
301     builtup
Name: Landcover, Length: 1053, dtype: object

In [ ]:
y_train, label_mapping = df_ops.convert_labels_to_integers(y_train)
y_val, label_mapping = df_ops.convert_labels_to_integers(y_val) 
y_test, label_mapping = df_ops.convert_labels_to_integers(y_test) 

In [ ]:
label_mapping

{'builtup': 0, 'grass': 1, 'road': 2, 'trees': 3}

# EXPORT

In [ ]:
y_train.to_csv("csv_files/y_train.csv", index=False)
y_val.to_csv("csv_files/y_val.csv", index=False)
y_test.to_csv("csv_files/y_test.csv", index=False)

END